# 05. Tuning and Ensembling
Цель этого ноутбука:

1. подобрать гиперпараметры для линейных моделей;
2. проверить отдельный preprocessing для деревьев;
3. попробовать простые ансамбли;
4. подготовить основу для дальнейшего сравнения с XGBoost, LightGBM и CatBoost.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

PROJECT_ROOT

PosixPath('/home/dany/ml_projects/house-price-regression')

In [3]:
from src.data import load_train_test
from src.features import prepare_features
from src.utils import (
    evaluate_model,
    create_submission,
    save_experiment_result,
)

In [4]:
train, test = load_train_test()

train.shape, test.shape

((1460, 81), (1459, 80))

In [5]:
X, y, X_test = prepare_features(train, test)

X.shape, y.shape, X_test.shape

((1460, 86), (1460,), (1459, 86))

In [6]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns

len(numeric_features), len(categorical_features)

(43, 43)

## Preprocessing для линейных моделей

Для линейных моделей оставляем preprocessing из baseline:

- числовые признаки:
  - заполнение пропусков медианой;
  - масштабирование через `StandardScaler`;
- категориальные признаки:
  - заполнение пропусков самым частым значением;
  - `OneHotEncoder`.


In [7]:
linear_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

linear_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", linear_numeric_transformer, numeric_features),
        ("cat", linear_categorical_transformer, categorical_features),
    ],
    sparse_threshold=0,
)

## Preprocessing для tree-based моделей

Для деревьев и бустингов используем отдельный preprocessing:

- числовые признаки:
  - заполнение пропусков медианой;
  - без масштабирования;
- категориальные признаки:
  - заполнение пропусков самым частым значением;
  - `OrdinalEncoder`.

Деревьям не нужно масштабирование числовых признаков. Кроме того, one-hot encoding может сильно раздувать признаковое пространство, поэтому для первого tree-specific эксперимента используем ordinal encoding.

In [8]:
tree_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

tree_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", tree_numeric_transformer, numeric_features),
        ("cat", tree_categorical_transformer, categorical_features),
    ]
)

## Cross-validation setup

Для честного сравнения новых экспериментов с предыдущими используем тот же подход, что и раньше:

- 5-fold KFold;
- `shuffle=True`;
- `random_state=42`;
- метрика — RMSE по `log1p(SalePrice)`.

In [9]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

## Ridge tuning

Ridge уже показывает лучший Kaggle score в проекте.

Теперь проверим несколько значений `alpha`, чтобы понять, можно ли улучшить качество через более аккуратный подбор регуляризации.

In [10]:
ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", linear_preprocessor),
        ("model", Ridge()),
    ]
)

ridge_param_grid = {
    "model__alpha": [
        0.1,
        0.3,
        1,
        3,
        5,
        10,
        15,
        20,
        30,
        50,
        100,
    ]
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

ridge_grid.fit(X, y)

ridge_best_rmse = -ridge_grid.best_score_
ridge_best_params = ridge_grid.best_params_

ridge_best_rmse, ridge_best_params

(np.float64(0.14702361890863005), {'model__alpha': 20})

## Lasso tuning

Lasso дала Kaggle score, близкий к Ridge.

Lasso использует L1-регуляризацию, поэтому может занулять часть коэффициентов и выполнять неявный отбор признаков.

In [11]:
lasso_pipeline = Pipeline(
    steps=[
        ("preprocessor", linear_preprocessor),
        ("model", Lasso(max_iter=50000, random_state=42)),
    ]
)

lasso_param_grid = {
    "model__alpha": [
        0.0001,
        0.0003,
        0.0005,
        0.0007,
        0.001,
        0.003,
        0.005,
    ]
}

lasso_grid = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

lasso_grid.fit(X, y)

lasso_best_rmse = -lasso_grid.best_score_
lasso_best_params = lasso_grid.best_params_

lasso_best_rmse, lasso_best_params

(np.float64(0.14170916758567775), {'model__alpha': 0.0005})

## ElasticNet tuning

ElasticNet объединяет L1 и L2-регуляризацию.

Это промежуточный вариант между Ridge и Lasso. Проверим несколько значений `alpha` и `l1_ratio`.

In [12]:
elasticnet_pipeline = Pipeline(
    steps=[
        ("preprocessor", linear_preprocessor),
        ("model", ElasticNet(max_iter=50000, random_state=42)),
    ]
)

elasticnet_param_grid = {
    "model__alpha": [
        0.0001,
        0.0003,
        0.0005,
        0.0007,
        0.001,
        0.003,
    ],
    "model__l1_ratio": [
        0.2,
        0.5,
        0.8,
        0.9,
    ],
}

elasticnet_grid = GridSearchCV(
    estimator=elasticnet_pipeline,
    param_grid=elasticnet_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
)

elasticnet_grid.fit(X, y)

elasticnet_best_rmse = -elasticnet_grid.best_score_
elasticnet_best_params = elasticnet_grid.best_params_

elasticnet_best_rmse, elasticnet_best_params

(np.float64(0.14172324346936085),
 {'model__alpha': 0.0005, 'model__l1_ratio': 0.9})

## Повторная оценка tuned linear models

После подбора гиперпараметров дополнительно оценим лучшие модели через общий `evaluate_model()`.

Это нужно, чтобы для всех экспериментов в `reports/model_results.csv` были заполнены и `CV RMSE`, и `CV std`. Так таблица экспериментов остаётся сопоставимой: baseline, tuned-модели, tree-prep модели и ансамбли оцениваются в одном формате.

In [13]:
ridge_tuned_cv_rmse, ridge_tuned_cv_std, _ = evaluate_model(
    ridge_grid.best_estimator_,
    X,
    y,
    cv,
)

lasso_tuned_cv_rmse, lasso_tuned_cv_std, _ = evaluate_model(
    lasso_grid.best_estimator_,
    X,
    y,
    cv,
)

elasticnet_tuned_cv_rmse, elasticnet_tuned_cv_std, _ = evaluate_model(
    elasticnet_grid.best_estimator_,
    X,
    y,
    cv,
)

tuned_linear_cv_results = pd.DataFrame(
    [
        {
            "Experiment": "011",
            "Model": "Ridge tuned",
            "CV RMSE": round(ridge_tuned_cv_rmse, 5),
            "CV std": round(ridge_tuned_cv_std, 5),
            "Best params": ridge_grid.best_params_,
        },
        {
            "Experiment": "012",
            "Model": "Lasso tuned",
            "CV RMSE": round(lasso_tuned_cv_rmse, 5),
            "CV std": round(lasso_tuned_cv_std, 5),
            "Best params": lasso_grid.best_params_,
        },
        {
            "Experiment": "013",
            "Model": "ElasticNet tuned",
            "CV RMSE": round(elasticnet_tuned_cv_rmse, 5),
            "CV std": round(elasticnet_tuned_cv_std, 5),
            "Best params": elasticnet_grid.best_params_,
        },
    ]
).sort_values("CV RMSE")

tuned_linear_cv_results

,Experiment,Model,CV RMSE,CV std,Best params
1,012,Lasso tuned,0.14171,0.04326,{'model__alpha': 0.0005}
2,013,ElasticNet tuned,0.14172,0.04305,"{'model__alpha': 0.0005, 'model__l1_ratio': 0.9}"
0,011,Ridge tuned,0.14702,0.04052,{'model__alpha': 20}


## Submission для tuned linear models

Создадим submission-файлы для tuned Ridge, Lasso и ElasticNet.

Kaggle score добавим вручную после отправки файлов.

In [14]:
tuned_linear_models = {
    "Ridge tuned": {
        "experiment": "011",
        "model": ridge_grid.best_estimator_,
        "filename": "submission_011_ridge_tuned.csv",
    },
    "Lasso tuned": {
        "experiment": "012",
        "model": lasso_grid.best_estimator_,
        "filename": "submission_012_lasso_tuned.csv",
    },
    "ElasticNet tuned": {
        "experiment": "013",
        "model": elasticnet_grid.best_estimator_,
        "filename": "submission_013_elasticnet_tuned.csv",
    },
}

In [15]:
for model_name, config in tuned_linear_models.items():
    print(f"Создание submission для модели: {model_name}")

    submission = create_submission(
        model=config["model"],
        X_test=X_test,
        test_ids=test["Id"],
        output_path=PROJECT_ROOT / "submissions" / config["filename"],
    )

    print(f"Submission сохранён: {config['filename']}")
    print("-" * 50)

Создание submission для модели: Ridge tuned
Submission сохранён: submission_011_ridge_tuned.csv
--------------------------------------------------
Создание submission для модели: Lasso tuned
Submission сохранён: submission_012_lasso_tuned.csv
--------------------------------------------------
Создание submission для модели: ElasticNet tuned
Submission сохранён: submission_013_elasticnet_tuned.csv
--------------------------------------------------


In [16]:
kaggle_scores = {
    "011": 0.13261,  # Ridge tuned
    "012": 0.13277,  # Lasso tuned
    "013": 0.13286,  # ElasticNet tuned
    "014": 0.14289,  # RandomForestRegressor tree-prep
    "015": 0.13395,  # GradientBoostingRegressor tree-prep
    "017": 0.13192,  # Ridge + Lasso ensemble
}

In [ ]:
results_path = PROJECT_ROOT / "reports" / "model_results.csv"

results = None

for _, row in tuned_linear_cv_results.iterrows():
    results = save_experiment_result(
        results_path=results_path,
        experiment=row["Experiment"],
        model_name=row["Model"],
        cv_rmse=row["CV RMSE"],
        cv_std=row["CV std"],
        kaggle_score=kaggle_scores.get(row["Experiment"], np.nan),
        notes="tuned linear model",
    )

results

,Experiment,Model,CV RMSE,CV std,Kaggle score,Notes
0,001,Ridge,0.14679,0.03932,0.13375,baseline
1,002,Ridge,0.11489,0.00818,0.13435,feature engineering + outlier removal
2,003,Ridge,0.14711,0.03976,0.13360,feature engineering without outlier removal
3,004,Ridge tuned,0.14700,0.03976,0.13262,feature engineering + tuned alpha without outl...
4,005,Ridge,0.14702,0.04052,0.13261,model comparison with feature engineering
5,006,Lasso,0.14171,0.04326,0.13277,model comparison with feature engineering
6,007,ElasticNet,0.14300,0.04132,0.13429,model comparison with feature engineering
7,008,RandomForestRegressor,0.14206,0.01889,0.14325,model comparison with feature engineering
8,009,HistGradientBoostingRegressor,0.13162,0.01786,NaN,model comparison with feature engineering
9,010,GradientBoostingRegressor,0.13224,0.01683,0.13680,model comparison with feature engineering


## Tree-specific preprocessing experiment

В предыдущем ноутбуке `HistGradientBoostingRegressor` показал хороший локальный CV, но на текущем представлении данных его финальное обучение оказалось нестабильно долгим даже при ограничении `max_iter`, `max_leaf_nodes` и включённом `early_stopping`.

Проблема связана не с размером данных, а с сочетанием модели и текущего представления признаков после preprocessing.

Поэтому в этом ноутбуке `HistGradientBoostingRegressor` временно исключён из экспериментов. Вместо этого проверяются:

- `RandomForestRegressor`;
- `GradientBoostingRegressor`.

In [18]:
tree_models = {
    "RandomForestRegressor tree-prep": RandomForestRegressor(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoostingRegressor tree-prep": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ),
}

In [19]:
tree_results = []

for model_name, model in tree_models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            ("model", model),
        ]
    )

    cv_rmse, cv_std, scores = evaluate_model(
        pipeline,
        X,
        y,
        cv,
    )

    tree_results.append(
        {
            "Model": model_name,
            "CV RMSE": round(cv_rmse, 5),
            "CV std": round(cv_std, 5),
        }
    )

    print(model_name)
    print(f"CV RMSE: {cv_rmse:.5f}")
    print(f"CV std:  {cv_std:.5f}")
    print("-" * 40)

RandomForestRegressor tree-prep
CV RMSE: 0.14160
CV std:  0.01904
----------------------------------------
GradientBoostingRegressor tree-prep
CV RMSE: 0.13120
CV std:  0.01600
----------------------------------------


In [20]:
tree_results_df = (
    pd.DataFrame(tree_results)
    .sort_values("CV RMSE")
    .reset_index(drop=True)
)

tree_results_df

,Model,CV RMSE,CV std
0,GradientBoostingRegressor tree-prep,0.1312,0.01600
1,RandomForestRegressor tree-prep,0.1416,0.01904


In [21]:
tree_experiment_ids = {
    "RandomForestRegressor tree-prep": "014",
    "GradientBoostingRegressor tree-prep": "015",
}

## Submission для всех tree-prep моделей

Создадим отдельный submission-файл для каждой tree-based модели с tree-specific preprocessing.

Это позволит сравнить не только локальный CV, но и Kaggle Public Leaderboard score для каждого варианта.

In [22]:
tree_submission_filenames = {
    "RandomForestRegressor tree-prep": "submission_014_random_forest_tree_prep.csv",
    "GradientBoostingRegressor tree-prep": "submission_015_gradient_boosting_tree_prep.csv",
}

In [23]:
tree_trained_pipelines = {}

for model_name, filename in tree_submission_filenames.items():
    print(f"Обучение модели: {model_name}")

    model = tree_models[model_name]

    pipeline = Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            ("model", model),
        ]
    )

    pipeline.fit(X, y)

    submission = create_submission(
        model=pipeline,
        X_test=X_test,
        test_ids=test["Id"],
        output_path=PROJECT_ROOT / "submissions" / filename,
    )

    tree_trained_pipelines[model_name] = pipeline

    print(f"Submission сохранён: {filename}")
    print("-" * 50)

Обучение модели: RandomForestRegressor tree-prep
Submission сохранён: submission_014_random_forest_tree_prep.csv
--------------------------------------------------
Обучение модели: GradientBoostingRegressor tree-prep
Submission сохранён: submission_015_gradient_boosting_tree_prep.csv
--------------------------------------------------


In [24]:
for model_name, experiment_id in tree_experiment_ids.items():
    row = tree_results_df[
        tree_results_df["Model"] == model_name
    ].iloc[0]

    results = save_experiment_result(
        results_path=results_path,
        experiment=experiment_id,
        model_name=model_name,
        cv_rmse=row["CV RMSE"],
        cv_std=row["CV std"],
        kaggle_score=kaggle_scores.get(experiment_id, np.nan),
        notes="tree-specific preprocessing",
    )

results

,Experiment,Model,CV RMSE,CV std,Kaggle score,Notes
0,001,Ridge,0.14679,0.03932,0.13375,baseline
1,002,Ridge,0.11489,0.00818,0.13435,feature engineering + outlier removal
2,003,Ridge,0.14711,0.03976,0.13360,feature engineering without outlier removal
3,004,Ridge tuned,0.14700,0.03976,0.13262,feature engineering + tuned alpha without outl...
4,005,Ridge,0.14702,0.04052,0.13261,model comparison with feature engineering
5,006,Lasso,0.14171,0.04326,0.13277,model comparison with feature engineering
6,007,ElasticNet,0.14300,0.04132,0.13429,model comparison with feature engineering
7,008,RandomForestRegressor,0.14206,0.01889,0.14325,model comparison with feature engineering
8,009,HistGradientBoostingRegressor,0.13162,0.01786,NaN,model comparison with feature engineering
9,010,GradientBoostingRegressor,0.13224,0.01683,0.13680,model comparison with feature engineering


# Простой ансамбль Ridge + Lasso

Ridge и Lasso дают близкие Kaggle scores.

Попробуем простой ансамбль: усредним предсказания tuned Ridge и tuned Lasso.

Важно: ансамбль нельзя оценивать только по Kaggle Public Leaderboard. Чтобы не выбирать лучшую модель только по Public LB, дополнительно посчитаем локальную OOF-оценку ансамбля.

## OOF-оценка ансамбля Ridge + Lasso

Submission ансамбля создаётся усреднением предсказаний в исходном масштабе цены.

Поэтому локальную OOF-оценку считаем тем же способом:

1. получаем OOF-предсказания Ridge в log-space;
2. получаем OOF-предсказания Lasso в log-space;
3. переводим оба набора предсказаний обратно в цену через `expm1`;
4. усредняем цены;
5. возвращаем ансамбль в log-space через `log1p`;
6. считаем RMSE относительно `log1p(SalePrice)`.

Так локальная оценка соответствует тому, как был создан Kaggle submission.

In [25]:
ridge_oof_log = cross_val_predict(
    ridge_grid.best_estimator_,
    X,
    y,
    cv=cv,
    n_jobs=-1,
)

lasso_oof_log = cross_val_predict(
    lasso_grid.best_estimator_,
    X,
    y,
    cv=cv,
    n_jobs=-1,
)

ridge_oof_price = np.expm1(ridge_oof_log)
lasso_oof_price = np.expm1(lasso_oof_log)

ensemble_oof_price = (
    0.5 * ridge_oof_price
    + 0.5 * lasso_oof_price
)

ensemble_oof_log = np.log1p(ensemble_oof_price)

ensemble_oof_rmse = root_mean_squared_error(
    y,
    ensemble_oof_log,
)

print(f"OOF RMSE ансамбля Ridge + Lasso: {ensemble_oof_rmse:.5f}")

OOF RMSE ансамбля Ridge + Lasso: 0.14922


## Fold-level CV для ансамбля

`cross_val_predict()` даёт общую OOF-оценку, но не показывает разброс качества между folds.

Поэтому дополнительно посчитаем RMSE ансамбля отдельно на каждом fold и получим `CV std`.

In [26]:
ensemble_fold_scores = []

for train_idx, valid_idx in cv.split(X):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    ridge_fold_model = clone(ridge_grid.best_estimator_)
    lasso_fold_model = clone(lasso_grid.best_estimator_)

    ridge_fold_model.fit(X_train, y_train)
    lasso_fold_model.fit(X_train, y_train)

    ridge_valid_log = ridge_fold_model.predict(X_valid)
    lasso_valid_log = lasso_fold_model.predict(X_valid)

    ridge_valid_price = np.expm1(ridge_valid_log)
    lasso_valid_price = np.expm1(lasso_valid_log)

    ensemble_valid_price = (
        0.5 * ridge_valid_price
        + 0.5 * lasso_valid_price
    )

    ensemble_valid_log = np.log1p(ensemble_valid_price)

    fold_rmse = root_mean_squared_error(
        y_valid,
        ensemble_valid_log,
    )

    ensemble_fold_scores.append(fold_rmse)

ensemble_cv_rmse = np.mean(ensemble_fold_scores)
ensemble_cv_std = np.std(ensemble_fold_scores)

print(f"CV RMSE ансамбля: {ensemble_cv_rmse:.5f}")
print(f"CV std ансамбля:  {ensemble_cv_std:.5f}")

CV RMSE ансамбля: 0.14307
CV std ансамбля:  0.04241


In [27]:
ridge_submission_path = PROJECT_ROOT / "submissions" / "submission_011_ridge_tuned.csv"
lasso_submission_path = PROJECT_ROOT / "submissions" / "submission_012_lasso_tuned.csv"

ridge_submission = pd.read_csv(ridge_submission_path)
lasso_submission = pd.read_csv(lasso_submission_path)

ensemble_submission = ridge_submission.copy()
ensemble_submission["SalePrice"] = (
    0.5 * ridge_submission["SalePrice"]
    + 0.5 * lasso_submission["SalePrice"]
)

ensemble_submission.head()

,Id,SalePrice
0,1461,116025.465339
1,1462,147249.072961
2,1463,171863.866159
3,1464,196733.667533
4,1465,194617.104769


In [28]:
ensemble_path = PROJECT_ROOT / "submissions" / "submission_017_ridge_lasso_ensemble.csv"

ensemble_submission.to_csv(ensemble_path, index=False)

ensemble_path

PosixPath('/home/dany/ml_projects/house-price-regression/submissions/submission_017_ridge_lasso_ensemble.csv')

In [29]:
results = save_experiment_result(
    results_path=results_path,
    experiment="017",
    model_name="Ridge + Lasso ensemble",
    cv_rmse=round(ensemble_cv_rmse, 5),
    cv_std=round(ensemble_cv_std, 5),
    kaggle_score=kaggle_scores.get("017", np.nan),
    notes="0.5 Ridge tuned + 0.5 Lasso tuned",
)

results

,Experiment,Model,CV RMSE,CV std,Kaggle score,Notes
0,001,Ridge,0.14679,0.03932,0.13375,baseline
1,002,Ridge,0.11489,0.00818,0.13435,feature engineering + outlier removal
2,003,Ridge,0.14711,0.03976,0.13360,feature engineering without outlier removal
3,004,Ridge tuned,0.14700,0.03976,0.13262,feature engineering + tuned alpha without outl...
4,005,Ridge,0.14702,0.04052,0.13261,model comparison with feature engineering
5,006,Lasso,0.14171,0.04326,0.13277,model comparison with feature engineering
6,007,ElasticNet,0.14300,0.04132,0.13429,model comparison with feature engineering
7,008,RandomForestRegressor,0.14206,0.01889,0.14325,model comparison with feature engineering
8,009,HistGradientBoostingRegressor,0.13162,0.01786,NaN,model comparison with feature engineering
9,010,GradientBoostingRegressor,0.13224,0.01683,0.13680,model comparison with feature engineering


## Сравнение CV и Kaggle Public Leaderboard

Сравним локальную CV-оценку и Kaggle Public Leaderboard score.

Это важно, потому что лучший результат по локальной CV не всегда совпадает с лучшим результатом на Public Leaderboard.

In [30]:
comparison = pd.DataFrame(
    [
        {
            "Model": "Ridge tuned",
            "CV RMSE": round(ridge_tuned_cv_rmse, 5),
            "Kaggle score": kaggle_scores["011"],
        },
        {
            "Model": "Lasso tuned",
            "CV RMSE": round(lasso_tuned_cv_rmse, 5),
            "Kaggle score": kaggle_scores["012"],
        },
        {
            "Model": "ElasticNet tuned",
            "CV RMSE": round(elasticnet_tuned_cv_rmse, 5),
            "Kaggle score": kaggle_scores["013"],
        },
        {
            "Model": "RandomForest tree-prep",
            "CV RMSE": tree_results_df.loc[
                tree_results_df["Model"] == "RandomForestRegressor tree-prep",
                "CV RMSE",
            ].iloc[0],
            "Kaggle score": kaggle_scores["014"],
        },
        {
            "Model": "GradientBoosting tree-prep",
            "CV RMSE": tree_results_df.loc[
                tree_results_df["Model"] == "GradientBoostingRegressor tree-prep",
                "CV RMSE",
            ].iloc[0],
            "Kaggle score": kaggle_scores["015"],
        },
        {
            "Model": "Ridge + Lasso ensemble",
            "CV RMSE": round(ensemble_cv_rmse, 5),
            "Kaggle score": kaggle_scores["017"],
        },
    ]
)

comparison

,Model,CV RMSE,Kaggle score
0,Ridge tuned,0.14702,0.13261
1,Lasso tuned,0.14171,0.13277
2,ElasticNet tuned,0.14172,0.13286
3,RandomForest tree-prep,0.14160,0.14289
4,GradientBoosting tree-prep,0.13120,0.13395
5,Ridge + Lasso ensemble,0.14307,0.13192


## Вывод по расхождению CV и Kaggle

В этом эксперименте видно, что локальная CV и Kaggle Public Leaderboard расходятся.

`GradientBoostingRegressor` с tree-specific preprocessing показывает сильный локальный CV RMSE, но его Kaggle score хуже, чем у ансамбля Ridge + Lasso.

Ансамбль Ridge + Lasso дал лучший Public Leaderboard score, но этот результат нельзя считать гарантированным улучшением обобщающей способности модели только на основании Public Leaderboard.

Для более надёжной оценки нужно:

- использовать OOF-предсказания;
- проверять несколько разных `random_state` для KFold;
- сравнивать результаты на CV и Kaggle вместе;
- избегать выбора модели только по Public Leaderboard.

Главный вывод: бустинг нельзя считать плохим только из-за текущего Kaggle score. Скорее, ему нужен отдельный preprocessing, tuning и более аккуратная валидация. Ансамбль линейных моделей оказался наиболее устойчивым на Public Leaderboard в текущем setup.

## Итоги tuning and ensembling

На этом этапе были проверены три направления улучшений:

1. tuning линейных моделей;
2. отдельный preprocessing для tree-based моделей;
3. простой ансамбль Ridge + Lasso.

### Kaggle results

| Experiment | Model | Kaggle score |
|---|---|---:|
| 011 | Ridge tuned | 0.13261 |
| 012 | Lasso tuned | 0.13277 |
| 013 | ElasticNet tuned | 0.13286 |
| 014 | RandomForestRegressor tree-prep | 0.14289 |
| 015 | GradientBoostingRegressor tree-prep | 0.13395 |
| 017 | Ridge + Lasso ensemble | 0.13192 |

### Основные наблюдения

Лучший результат на Kaggle дал простой ансамбль Ridge + Lasso: `0.13192`.

Ансамбль был дополнительно проверен локально через OOF-предсказания.

Tuned Ridge и tuned Lasso по отдельности почти не улучшили результат относительно предыдущих экспериментов.

Tree-specific preprocessing немного улучшил GradientBoostingRegressor относительно предыдущего sklearn boosting эксперимента, но всё ещё не превзошёл линейные модели на Kaggle.

RandomForestRegressor остался слабым вариантом для этой задачи.

### Методологический вывод

Локальная CV и Public Leaderboard расходятся.

По локальной CV сильным выглядит GradientBoostingRegressor с tree-specific preprocessing, но на Kaggle лучший результат показывает ансамбль Ridge + Lasso.

Бустинг требует отдельной настройки, а ансамбль линейных моделей оказался наиболее устойчивым на Public Leaderboard в текущем признаковом пространстве.

### Важная оговорка

Ансамбль дал лучший Public Leaderboard score, но этот результат нельзя считать гарантированным улучшением обобщающей способности модели.

Для более надёжной оценки нужно:

- проверять ансамбль через OOF-предсказания;
- использовать несколько разных `random_state` для KFold;
- сравнивать результаты не только по Kaggle Public Leaderboard, но и по локальной валидации.

### Следующие шаги

- проверить разные веса Ridge/Lasso;
- попробовать log-space ensembling;
- попробовать ансамбль Ridge + Lasso + ElasticNet;
- добавить XGBoost, LightGBM и CatBoost;
- создать `src/train.py` для воспроизводимого запуска лучшего пайплайна;
- сохранить лучшую модель в `models/`.